<a href="https://colab.research.google.com/github/novikovamaria137-png/mtuci-llm-course/blob/main/lesson-2.6/practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Открыть в Colab"/></a>

# Практика 2.6. Эмбеддинги и семантический поиск

**Модуль 2 · Урок 6 · 110 минут**

До сих пор модель порождала текст. Сегодня — вторая её задача: превращать текст в числа, по которым можно искать похожее по смыслу.

---

### Что вы сделаете

| Шаг | Что делаем | Время | Нужен ключ |
|---|---|---|---|
| 0 | Восстановим каркас | 5 мин | нет |
| 1 | Разберём арифметику: нормализация и косинус | 20 мин | нет |
| 2 | Соберём поиск ближайших с порогом отсечки | 20 мин | нет |
| 3 | Напишем учебный эмбеддер без сети и модели | 20 мин | нет |
| 4 | Увидим границу: чего он НЕ умеет | 20 мин | нет |
| 5 | Применим поиск к задачам: дубликаты, группировка | 15 мин | нет |
| 6 | Получим настоящие эмбеддинги через API | 10 мин | да |

> **Шаг 4 — главный в уроке.** Учебный эмбеддер работает без сети и позволяет проверить всю арифметику. Но он ловит совпадение символов, а не смысл. Увидев это своими глазами, вы поймёте, за что именно платите, обращаясь к настоящей модели.

---
## Шаг 0. Каркас

*Статус ячейки: проверено запуском.*

In [ ]:
import sys
from pathlib import Path

ROOT = Path("llm-project")
(ROOT / "llmcourse").mkdir(parents=True, exist_ok=True)
(ROOT / "llmcourse" / "__init__.py").write_text("", encoding="utf-8")
if str(ROOT.resolve()) not in sys.path:
    sys.path.insert(0, str(ROOT.resolve()))

print("Проект:", ROOT.resolve())
print("Внешних зависимостей на шагах 0-5 нет: всё считается стандартной библиотекой.")

---
## Шаг 1. Из чего состоит семантический поиск

Эмбеддинг — представление текста набором чисел, устроенное так, что близким по смыслу текстам соответствуют близкие наборы. Дальше поиск по смыслу сводится к поиску ближайших векторов, то есть к арифметике.

Задача распадается на две части, и путать их нельзя:

| Часть | Что делает | Кто отвечает |
|---|---|---|
| Модель эмбеддингов | превращает текст в вектор | внешний сервис или локальная модель |
| Арифметика поиска | находит ближайшие векторы | **ваш код** |

Вторая часть одинакова для любых векторов, откуда бы они ни взялись. Её мы и напишем — целиком и с проверками.

### Почему косинус, а не расстояние

Длинный текст даёт вектор большей длины. Если сравнивать обычным расстоянием, длина начнёт влиять на результат: длинные документы окажутся «дальше» от всего, включая то, о чём они и написаны.

Косинусная близость сравнивает **направление**, а не длину: это косинус угла между векторами. Единица — направления совпали, ноль — векторы независимы, минус единица — противоположны.

*Статус ячейки: проверено запуском, включая сверку с ручным расчётом.*

In [ ]:
%%writefile llm-project/llmcourse/embeddings.py
"""Эмбеддинги и семантический поиск. Урок 2.6.

Эмбеддинг — это представление текста набором чисел, устроенное так, что
близким по смыслу текстам соответствуют близкие наборы. Дальше поиск по
смыслу сводится к поиску ближайших векторов, то есть к арифметике.

Модуль делает две разные вещи, и их важно не путать:

  1. Арифметику поиска — нормализацию, косинусную близость, выбор ближайших.
     Она одинакова для любых векторов, откуда бы те ни взялись, и полностью
     проверяема.

  2. Учебный эмбеддер hash_embed — заглушку, работающую без сети и без
     модели. Он ловит совпадение символов, а не смысл. Нужен, чтобы
     проверить арифметику и увидеть своими глазами, чего именно не хватает
     без настоящей модели.

Подменять вторым первое нельзя. Границе посвящён отдельный шаг практики.
"""
import hashlib
import math
import re


class EmbeddingError(ValueError):
    """Ошибка в данных для векторных операций."""


# ── арифметика ────────────────────────────────────────────────────
def norm(v):
    """Длина вектора."""
    return math.sqrt(sum(x * x for x in v))


def normalize(v):
    """Приводит вектор к единичной длине.

    Нужна, чтобы длина не влияла на сравнение: длинный текст даёт вектор
    большей длины, но это не делает его «более похожим» на всё подряд.
    Нулевой вектор возвращается как есть — делить на ноль нельзя, а падать
    на пустом тексте модуль не должен.
    """
    n = norm(v)
    if n == 0:
        return list(v)
    return [x / n for x in v]


def cosine(a, b):
    """Косинусная близость: 1 — совпадение направления, 0 — независимость.

    Считается как скалярное произведение нормализованных векторов, то есть
    как косинус угла между ними. Сравнивается направление, а не длина.
    """
    if len(a) != len(b):
        raise EmbeddingError(
            f"векторы разной длины: {len(a)} и {len(b)}. "
            "Скорее всего, они получены разными моделями")
    na, nb = norm(a), norm(b)
    if na == 0 or nb == 0:
        return 0.0
    return sum(x * y for x, y in zip(a, b)) / (na * nb)


def search(query_vec, items, top_k=5, threshold=None):
    """Ближайшие элементы к запросу.

    items — последовательность пар (идентификатор, вектор).
    threshold — отсечка по близости; ниже неё результаты не возвращаются.

    Возвращает список пар (идентификатор, близость) по убыванию.
    """
    if top_k < 1:
        raise EmbeddingError("top_k должно быть не меньше 1")
    scored = [(key, cosine(query_vec, vec)) for key, vec in items]
    if threshold is not None:
        scored = [p for p in scored if p[1] >= threshold]
    scored.sort(key=lambda p: (-p[1], str(p[0])))
    return scored[:top_k]


# ── учебный эмбеддер ──────────────────────────────────────────────
_WORD = re.compile(r"\w+", re.UNICODE)


def _ngrams(text, n=3):
    """Символьные n-граммы слов с границами.

    Границы обозначены пробелами, чтобы начало и конец слова участвовали
    в сравнении: так «кот» и «икота» окажутся дальше друг от друга.
    """
    out = []
    for word in _WORD.findall(text.lower()):
        padded = f" {word} "
        for i in range(len(padded) - n + 1):
            out.append(padded[i:i + n])
    return out


def hash_embed(text, dim=256, n=3):
    """УЧЕБНЫЙ эмбеддер: хеширование символьных n-грамм.

    Это не языковая модель. Он не знает, что «врач» и «доктор» — про одно,
    потому что сравнивает написание, а не смысл. Зато он детерминирован,
    не требует сети и позволяет проверить всю арифметику поиска.

    Приём называется hashing trick и применяется по-настоящему — но для
    лексического поиска, а не семантического.
    """
    if dim < 8:
        raise EmbeddingError("размерность должна быть не меньше 8")
    vec = [0.0] * dim
    grams = _ngrams(text, n)
    if not grams:
        return vec
    for g in grams:
        h = hashlib.md5(g.encode("utf-8")).digest()
        idx = int.from_bytes(h[:4], "big") % dim
        sign = 1.0 if h[4] % 2 == 0 else -1.0   # знак снижает влияние коллизий
        vec[idx] += sign
    return normalize(vec)


def embed_all(texts, embed=hash_embed):
    """Векторы для списка текстов. embed заменяется на настоящую модель."""
    return [embed(t) for t in texts]


In [ ]:
import importlib
importlib.invalidate_caches()
import llmcourse.embeddings
importlib.reload(llmcourse.embeddings)
from llmcourse.embeddings import (norm, normalize, cosine, search,
                                  hash_embed, embed_all, EmbeddingError)

print("Нормализация — приведение к единичной длине:")
v = normalize([3, 4])
print(f"  [3, 4] -> {v}, длина {norm(v):.6f}")
print("  (проверьте вручную: 3/5 = 0.6, 4/5 = 0.8)")
print()

print("Косинусная близость:")
pairs = [([1, 0], [1, 0], "совпадают"),
         ([1, 0], [0, 1], "перпендикулярны"),
         ([1, 0], [-1, 0], "противоположны"),
         ([1, 2, 3], [10, 20, 30], "то же направление, длина в 10 раз больше")]
for a, b, label in pairs:
    print(f"  {str(a):<12} и {str(b):<14} -> {cosine(a, b):+.4f}   {label}")

print()
print("Последняя строка — самая важная. Длина вектора на результат не влияет,")
print("поэтому длинный документ не оказывается «дальше» просто из-за размера.")

print()
print("Крайние случаи, на которых обычно падают самодельные реализации:")
print("  нулевой вектор ->", cosine([0, 0], [1, 1]), "(а не деление на ноль)")
try:
    cosine([1, 2], [1, 2, 3])
except EmbeddingError as e:
    print("  разная длина  ->", e)

---
## Шаг 2. Поиск ближайших

Имея близость, поиск сводится к сортировке. Две детали, которые отличают рабочий поиск от учебного:

**Порог отсечки.** Без него поиск всегда вернёт `top_k` результатов — даже когда в базе нет ничего подходящего. Пользователь получит пять «ответов» на вопрос, ответа на который у вас нет.

**Устойчивый порядок при равной близости.** Иначе одинаковые запросы будут давать разный порядок выдачи, и вы не сможете ни воспроизвести жалобу, ни написать тест.

*Статус ячейки: проверено запуском.*

In [ ]:
items = [("а. точное совпадение", [1, 0]),
         ("б. почти то же",       [0.9, 0.1]),
         ("в. про другое",        [0, 1]),
         ("г. противоположное",   [-1, 0])]

print("Запрос: [1, 0]")
print()
for k, s in search([1, 0], items, top_k=4):
    print(f"  {s:+.4f}  {k}")

print()
print("С порогом 0.5 — отсекается всё, что не похоже:")
for k, s in search([1, 0], items, top_k=4, threshold=0.5):
    print(f"  {s:+.4f}  {k}")

print()
res = search([1, 0], items, top_k=4, threshold=0.99)
print(f"С порогом 0.99 найдено результатов: {len(res)}")
print("Пустой результат — это тоже ответ. Честнее выдать его,")
print("чем показать пять непохожих документов.")

print()
tie = [("яблоко", [1, 0]), ("апельсин", [1, 0]), ("банан", [1, 0])]
print("Порядок при одинаковой близости:", [k for k, _ in search([1, 0], tie, top_k=3)])
print("Он устойчив — при повторном запуске будет тот же.")

In [ ]:
# Проверка на подсаженном документе: работает ли поиск на объёме.
import random
random.seed(0)

corpus = [(f"doc{i}", [random.gauss(0, 1) for _ in range(64)]) for i in range(300)]
planted = [random.gauss(0, 1) for _ in range(64)]
corpus[137] = ("doc137", planted)

top = search(planted, corpus, top_k=3)
print("Ищем документ, который сами же положили в базу под номером 137:")
for k, s in top:
    print(f"  {s:+.4f}  {k}")

assert top[0][0] == "doc137" and abs(top[0][1] - 1.0) < 1e-12
print()
print("[ok] найден первым, близость ровно 1.0000")
print()

# Насколько похожими бывают заведомо случайные документы?
import statistics
noise = [cosine(planted, v) for k, v in corpus if k != "doc137"]
print(f"Близость к 299 случайным документам:")
print(f"  среднее      {statistics.mean(noise):+.4f}")
print(f"  разброс      {statistics.pstdev(noise):.4f}  (теоретически 1/sqrt(64) = {1/8:.4f})")
print(f"  максимум     {max(noise):+.4f}")
print()
print("Вот это и есть причина, по которой нужен порог отсечки.")
print("Случайный шум даёт близость до 0.37 — значит, порог 0.3 пропустит мусор.")
print("Порог подбирают выше уровня шума, а уровень шума зависит от размерности:")
print("чем она больше, тем случайные векторы ближе к перпендикулярным.")

---
## Шаг 3. Учебный эмбеддер

Арифметику проверили. Теперь нужны векторы из настоящих текстов — а модели у нас нет.

Напишем заглушку: разобьём слова на трёхсимвольные куски, каждый кусок захешируем и по хешу выберем позицию в векторе. Приём называется hashing trick и применяется по-настоящему — но для **лексического** поиска, а не семантического.

**Это не языковая модель.** Она сравнивает написание, а не смысл. Зачем она нужна:

- работает без сети, без ключа и без GPU;
- детерминирована — на одном тексте всегда один вектор, значит, на ней можно писать тесты;
- позволяет проверить всю арифметику поиска на настоящих текстах;
- а на шаге 4 покажет, чего именно не хватает без настоящей модели.

*Статус ячейки: проверено запуском.*

In [ ]:
docs = [
    "Договор аренды нежилого помещения",
    "Договор аренды офиса в бизнес-центре",
    "Договор поставки оборудования",
    "Акт приёма-передачи помещения",
    "Прогноз погоды на завтра в Москве",
    "Инструкция по подключению принтера",
]

vectors = embed_all(docs)
print(f"Векторов: {len(vectors)}, размерность каждого: {len(vectors[0])}")
print(f"Длина первого вектора: {norm(vectors[0]):.6f} (нормализован)")
print()

a, b = hash_embed("Договор аренды помещения"), hash_embed("Договор аренды помещения")
print("Детерминированность:", "одинаковые векторы" if a == b else "РАЗНЫЕ — это ошибка")
print("Пустой текст:", "нулевой вектор" if hash_embed("", dim=64) == [0.0]*64 else "не нулевой")

print()
query = "аренда помещения"
qv = hash_embed(query)
print(f"Запрос: {query!r}")
print()
for k, s in search(qv, list(zip(docs, vectors)), top_k=6):
    mark = "  <-- нашлось" if s > 0.3 else ""
    print(f"  {s:+.4f}  {k}{mark}")

print()
print("Поиск работает: договоры аренды наверху, погода и принтер внизу.")
print("Но работает он по совпадению букв. В этом и подвох — см. шаг 4.")

---
## Шаг 4. Граница: чего заглушка не умеет

Главный шаг урока.

Ниже — три пары слов. Посмотрите на числа и подумайте, что они означают, прежде чем читать вывод под ячейкой.

*Статус ячейки: проверено запуском.*

In [ ]:
pairs = [
    ("врач", "доктор",     "синонимы: разное написание, один смысл"),
    ("врач", "врачебный",  "один корень, разный смысл"),
    ("кот",  "кошка",      "близкие по смыслу, частично разное написание"),
    ("кот",  "ток",        "те же буквы, смысла общего нет"),
    ("замок на двери", "замок на горе", "одно написание, разные смыслы"),
]

print(f"{'пара':<34} {'близость':>9}   комментарий")
print("-" * 88)
for a, b, note in pairs:
    s = cosine(hash_embed(a), hash_embed(b))
    print(f"{a + ' / ' + b:<34} {s:>+9.4f}   {note}")

print()
syn = cosine(hash_embed("врач"), hash_embed("доктор"))
lex = cosine(hash_embed("врач"), hash_embed("врачебный"))
print(f"Синонимы «врач/доктор»:        {syn:+.4f}")
print(f"Однокоренные «врач/врачебный»: {lex:+.4f}")
print()
print("Однокоренные ближе синонимов. Для поиска по смыслу это ровно наоборот")
print("тому, что нужно.")

### Что именно вы сейчас увидели

Заглушка считает похожими тексты, у которых **совпадают буквы**. Поэтому:

- **«врач» и «доктор»** — про одно, но написаны по-разному, и близость около нуля;
- **«врач» и «врачебный»** — про разное, но пишутся похоже, и близость высокая;
- **«замок на двери» и «замок на горе»** — вообще разные вещи, а написание одинаковое.

Настоящая модель эмбеддингов обучена так, что близость отражает смысл, а не написание. Именно за это вы платите, обращаясь к ней: не за скорость и не за размерность вектора, а за то, что «врач» и «доктор» окажутся рядом.

**Практический вывод.** Когда семантический поиск работает плохо, причина чаще всего не в арифметике — она у вас проверена и не ошибается. Причина либо в модели эмбеддингов, либо в том, как нарезаны документы. Второму посвящён урок 2.7.

**И честная оговорка в обратную сторону.** Лексический поиск не бесполезен. Артикулы, номера договоров, фамилии, коды ошибок ищутся именно по написанию, и семантическая модель здесь скорее мешает. Хорошие поисковые системы сочетают оба подхода — к этому курс вернётся в уроке 2.9.

---
## Шаг 5. Что ещё умеет близость векторов

Поиск — самое частое применение, но не единственное. Ещё три задачи решаются той же арифметикой.

| Задача | Как решается |
|---|---|
| Поиск похожего | ближайшие к вектору запроса |
| Поиск дубликатов | пары с близостью выше порога |
| Группировка | объединение близких между собой |
| Классификация | ближайший из векторов-образцов классов |

*Статус ячейки: проверено запуском.*

In [ ]:
# Дубликаты: пары, близость которых выше порога.
items = [
    "Заявление на отпуск с 1 июля",
    "Заявление на отпуск с 1 июля.",          # отличается точкой
    "Заявление на отпуск с 15 августа",
    "Служебная записка о закупке бумаги",
    "Служебная записка о закупке бумаги для принтера",
]
vecs = embed_all(items)

print("Возможные дубликаты (близость выше 0.85):")
print()
for i in range(len(items)):
    for j in range(i + 1, len(items)):
        s = cosine(vecs[i], vecs[j])
        if s > 0.85:
            print(f"  {s:.4f}")
            print(f"    - {items[i]}")
            print(f"    - {items[j]}")
            print()

print("Порог подбирается на своих данных: слишком высокий пропустит дубликаты,")
print("слишком низкий склеит разные документы. Это настройка, а не константа.")

In [ ]:
# Классификация по образцам: ближайший класс.
CLASSES = {
    "договор":   "договор соглашение стороны обязуются предмет договора",
    "заявление": "заявление прошу предоставить отпуск с сохранением",
    "инструкция":"инструкция порядок действий шаг подключение настройка",
}
class_vecs = {name: hash_embed(text) for name, text in CLASSES.items()}

tests = [
    "Договор оказания услуг между сторонами",
    "Прошу предоставить отпуск с 3 марта",
    "Порядок подключения оборудования: шаг первый",
    "Стихотворение про осень и листья",
]

print(f"{'документ':<45} {'класс':<12} близость")
print("-" * 74)
for t in tests:
    v = hash_embed(t)
    best = search(v, list(class_vecs.items()), top_k=1)[0]
    label = best[0] if best[1] > 0.15 else "не определён"
    print(f"{t[:44]:<45} {label:<12} {best[1]:+.4f}")

print()
print("Последняя строка — то, ради чего нужен порог. Без него стихотворение")
print("получило бы ярлык ближайшего класса, каким бы далёким тот ни был.")

---
## Шаг 6. Настоящие эмбеддинги

Единственный шаг с ключом. Заменяем заглушку на модель эмбеддингов и повторяем проверку с шага 4.

Что смотреть: станут ли «врач» и «доктор» ближе, чем «врач» и «врачебный». Если да — вы своими глазами увидели, что покупается вместе с моделью.

Обратите внимание: **весь остальной код не меняется**. Арифметика поиска, порог, дубликаты, классификация — всё работает поверх любых векторов. Это тот же принцип переносимости, что и в уроке 2.5.

*Статус ячейки: требует проверки на живом ключе. Автор материалов эту ячейку с настоящим ключом не запускал.*

In [ ]:
# Заготовка. Уточните способ получения эмбеддингов у своего поставщика:
# у разных сервисов различаются и имя модели, и вид ответа.
import os

HAVE_KEY = bool(os.environ.get("LLM_API_KEY", "").strip())

def real_embed(text):
    """Возвращает вектор от настоящей модели эмбеддингов."""
    from openai import OpenAI
    client = OpenAI(base_url=os.environ["LLM_BASE_URL"],
                    api_key=os.environ["LLM_API_KEY"])
    r = client.embeddings.create(model=os.environ.get("EMBED_MODEL", "укажите-модель"),
                                 input=text)
    return r.data[0].embedding

if not HAVE_KEY:
    print("Ключ не подключён — шаг пропускается.")
    print("Шаги 0-5 дают всё содержание урока: арифметика поиска не зависит")
    print("от того, откуда взялись векторы.")
else:
    print(f"{'пара':<34} {'заглушка':>10} {'модель':>10}")
    print("-" * 58)
    for a, b in [("врач", "доктор"), ("врач", "врачебный"),
                 ("кот", "кошка"), ("кот", "ток")]:
        s_fake = cosine(hash_embed(a), hash_embed(b))
        try:
            s_real = cosine(real_embed(a), real_embed(b))
            print(f"{a + ' / ' + b:<34} {s_fake:>+10.4f} {s_real:>+10.4f}")
        except Exception as e:
            print(f"{a + ' / ' + b:<34} {s_fake:>+10.4f}   ошибка: {type(e).__name__}")
            break
    print()
    print("Если у модели «врач/доктор» вышло выше, чем «врач/врачебный» —")
    print("вы увидели разницу между лексической и семантической близостью.")

---
## Задание

1. **Свои пары.** Придумайте пять пар слов из вашей предметной области: синонимы, однокоренные, омонимы. Прогоните через заглушку и запишите, где она ошибается. Это будет ваш тестовый набор для проверки настоящей модели.

2. **Подбор порога.** Возьмите 20 документов своей организации, посчитайте все попарные близости и подберите порог, при котором дубликаты находятся, а разные документы не склеиваются. Объясните письменно, почему выбрали именно это значение.

3. **Влияние размерности.** Прогоните поиск при размерности 32, 256 и 1024. Посмотрите, как меняются числа и меняется ли порядок выдачи. Объясните результат.

### Повышенной сложности

4. Добавьте в поиск учёт длины n-грамм: сравните результаты при n=2, 3, 4. Какое значение лучше работает на русском языке и почему?

5. Реализуйте поиск, который сочетает лексическую близость (заглушка) и семантическую (настоящая модель), объединяя два списка результатов. Сравните с каждым по отдельности.

6. Оцените, во что обойдётся получение эмбеддингов для 10 000 документов: используйте расчёт из урока 2.2 и тарифы своего поставщика.

---
## Чек-лист

- [ ] Могу объяснить, почему для сравнения векторов берут косинус, а не расстояние
- [ ] Знаю, что нулевой вектор и векторы разной длины не должны ронять поиск
- [ ] Понимаю, зачем в поиске нужен порог отсечки
- [ ] Своими глазами видел, что заглушка не сближает синонимы
- [ ] Могу объяснить, что покупается вместе с настоящей моделью эмбеддингов
- [ ] Знаю три задачи помимо поиска, решаемые той же арифметикой
- [ ] Понимаю, что лексический поиск не бесполезен и когда он лучше

## Частые проблемы

| Симптом | Причина и что делать |
|---|---|
| `EmbeddingError: векторы разной длины` | Векторы получены разными моделями. Пересчитайте базу целиком одной |
| Поиск всегда возвращает 5 результатов | Не задан порог отсечки. Пустой результат — тоже ответ |
| Синонимы не находятся | Ожидаемо для лексического поиска. Нужна модель эмбеддингов |
| Находится не то, хотя модель настоящая | Скорее всего, дело в нарезке документов — урок 2.7 |
| Все близости около нуля | Нормально для случайных векторов: в многомерном пространстве они почти перпендикулярны |
| Порядок выдачи меняется между запусками | Не задан устойчивый порядок при равной близости |

## Что дальше

Урок 2.7 — **документы и векторные базы**. Реальные документы не помещаются в один вектор: договор на 40 страниц придётся разрезать, и от того, как именно вы это сделаете, качество поиска зависит сильнее, чем от выбора модели.